# RegLens · Parallel-track de-risk: load a real pretrained ChromBPNet & score one variant

**Goal (run this on a Colab GPU today):** confirm that a *real* pretrained ChromBPNet
checkpoint loads and scores a single variant, and that its input/output contract matches
the assumptions baked into `reglens/tools/chrombpnet_score.py`. If reality differs from our
stub, we want to know **now**, not on Saturday.

This notebook is **self-contained**: it fetches the 2114 bp reference window from the UCSC
REST API (no multi-GB genome download) and inlines the exact one-hot / JSD logic RegLens
uses, so it runs on a bare Colab with just a couple of pip installs.

### The three ChromBPNet gotchas this notebook enforces
1. **Use the bias-corrected model** `chrombpnet_nobias.h5` — *not* `bias_model_scaled.h5`
   or the raw `chrombpnet.h5`. Loading the wrong one gives a garbage Δ.
2. **Confirm exact I/O**: one-hot `(N, 2114, 4)` in; two heads out — profile logits
   `(N, ~1000)` and a scalar logcount `(N, 1)`. We assert these shapes before trusting
   any number. Δ log-count is the primary signal; we also grab the **profile-shape change**
   (Jensen–Shannon distance) for Thursday's motif story.
3. **Motif effect (Thursday)**: ISM + JASPAR match is the right MVP — skip full TF-MoDISco.

### Example variant
`rs1427407` — **chr2:60,495,250 G>T (hg38)** in the BCL11A +58 kb erythroid enhancer; disrupts
a **GATA1** motif and de-represses fetal hemoglobin. Score it with an **erythroid / K562-like**
ChromBPNet model to see the expected accessibility drop. (Swap in your own variant + model.)

In [ ]:
# --- Dependencies -----------------------------------------------------------
# TensorFlow to load the Keras model; numpy + requests for scoring/fetching.
# (No 'chrombpnet' pip package is needed just to run inference on a saved model.)
!pip -q install "tensorflow>=2.12" numpy requests

import numpy as np
import requests
import tensorflow as tf
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# --- Config -----------------------------------------------------------------
# GOTCHA #1: we score with the BIAS-CORRECTED model `chrombpnet_nobias.h5`.
#
# MONEY-SHOT default: the K562 ATAC ChromBPNet model from the ChromBPNet manuscript
# (ENCODE annotation ENCSR467RSV, models file ENCFF984RAF). Same model family the
# paper used to score red-blood-cell-trait variants, so rs1427407 (BCL11A/GATA1)
# should show an accessibility DROP. Archive is ~727 MB and holds all 5 folds; we
# use fold 0. To use a different model, set MODEL_PATH to your own *_nobias.h5.
import glob, os, tarfile

MODEL_PATH = ""  # set to an existing *_nobias.h5 to skip the ENCODE download below
ENCODE_TAR_URL = "https://www.encodeproject.org/files/ENCFF984RAF/@@download/ENCFF984RAF.tar.gz"

if not MODEL_PATH:
    if not os.path.exists("ENCFF984RAF.tar.gz"):
        print("Downloading K562 ATAC ChromBPNet models (~727 MB, one-time)...")
        !wget -q -c -O ENCFF984RAF.tar.gz "$ENCODE_TAR_URL"
    with tarfile.open("ENCFF984RAF.tar.gz") as t:
        t.extractall("encode_models")
    hits = sorted(glob.glob("encode_models/**/*chrombpnet_nobias*.h5", recursive=True))
    assert hits, "No chrombpnet_nobias.h5 in archive - run `!find encode_models` to locate the .h5."
    MODEL_PATH = hits[0]  # fold 0
    print(f"Using {MODEL_PATH}  ({len(hits)} fold(s) found)")

# Sanity guard for GOTCHA #1: refuse the obviously-wrong model files by name.
_base = os.path.basename(MODEL_PATH).lower()
assert os.path.exists(MODEL_PATH), f"Model not found at {MODEL_PATH!r}."
if "bias_model" in _base or _base == "chrombpnet.h5":
    raise ValueError(
        f"{MODEL_PATH!r} looks like the bias / raw model. Use the bias-CORRECTED "
        "'chrombpnet_nobias.h5' for variant scoring (gotcha #1)."
    )

# The example variant (hg38). Change these three lines to score a different one.
CHROM, POS, REF, ALT = "chr2", 60495250, "G", "T"   # rs1427407, BCL11A enhancer
CELLTYPE = "K562 ATAC (erythroleukemia)"
INPUT_LEN = 2114                                     # ChromBPNet receptive field

In [ ]:
# --- Helpers (identical logic to reglens/tools/chrombpnet_score.py) ----------
_BASE_TO_INDEX = {"A": 0, "C": 1, "G": 2, "T": 3}

def one_hot_encode(seq: str) -> np.ndarray:
    """One-hot encode DNA as (L, 4), columns A,C,G,T; non-ACGT -> all-zero row."""
    seq = seq.upper()
    oh = np.zeros((len(seq), 4), dtype=np.float32)
    for i, b in enumerate(seq):
        j = _BASE_TO_INDEX.get(b)
        if j is not None:
            oh[i, j] = 1.0
    return oh

def softmax(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=-1, keepdims=True)

def jensen_shannon_distance(a: np.ndarray, b: np.ndarray) -> float:
    """JS distance in [0,1] between two profile-logit vectors (matches variant-scorer 'jsd')."""
    p, q = softmax(a.ravel().astype(np.float64)), softmax(b.ravel().astype(np.float64))
    m = 0.5 * (p + q)
    kl = lambda u, v: float(np.sum(u[u > 0] * np.log2(u[u > 0] / v[u > 0])))
    return float(np.sqrt(max(0.5 * kl(p, m) + 0.5 * kl(q, m), 0.0)))

def fetch_hg38_window(chrom: str, pos_1based: int, length: int) -> tuple[str, int]:
    """Fetch a `length`-bp hg38 window centered on a 1-based position via the UCSC API.

    Returns (sequence, variant_offset) with the variant base at index length//2.
    """
    half = length // 2
    start0 = (pos_1based - 1) - half           # 0-based inclusive
    end0 = start0 + length                     # 0-based exclusive
    url = (f"https://api.genome.ucsc.edu/getData/sequence?genome=hg38"
           f";chrom={chrom};start={start0};end={end0}")
    dna = requests.get(url, timeout=30).json()["dna"].upper()
    assert len(dna) == length, f"UCSC returned {len(dna)} bp, expected {length}"
    return dna, half

print("helpers ready")

In [ ]:
# --- Load the model and ASSERT the I/O contract (gotcha #2) ------------------
# compile=False: we only run inference, never optimize (avoids needing the custom
# ChromBPNet loss objects).
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

in_shape = model.input_shape
out_shapes = model.output_shape
print("input_shape :", in_shape)
print("output_shape:", out_shapes)

# Input must be (None, 2114, 4).
assert tuple(in_shape[1:]) == (INPUT_LEN, 4), (
    f"Expected input (None, {INPUT_LEN}, 4), got {in_shape}. If your model uses a "
    "different input length, set INPUT_LEN and window_length in RegLens to match."
)
# Two heads expected: [profile, counts].
assert isinstance(out_shapes, list) and len(out_shapes) == 2, (
    f"Expected 2 output heads [profile, counts], got {out_shapes}."
)
prof_shape, cnt_shape = out_shapes[0], out_shapes[1]
print(f"profile head: {prof_shape}  (expected (None, ~1000))")
print(f"counts head : {cnt_shape}   (expected (None, 1))")
assert cnt_shape[-1] == 1, f"counts head should be scalar per seq, got {cnt_shape}"
PROFILE_LEN = prof_shape[-1]
print("\n✔ I/O contract matches RegLens's KerasChromBPNetBackend assumptions.")

In [ ]:
# --- Fetch window, build ref/alt, score the variant -------------------------
ref_seq, offset = fetch_hg38_window(CHROM, POS, INPUT_LEN)

# GOTCHA (RegLens ref-check): confirm the genome base matches the declared REF.
observed = ref_seq[offset]
if observed != REF:
    print(f"⚠ Reference mismatch: hg38 has {observed!r} at {CHROM}:{POS}, variant declares "
          f"{REF!r}. Check the coordinate/strand. (Proceeding with the genome base.)")
alt_seq = ref_seq[:offset] + ALT + ref_seq[offset + 1:]

# Batch = [ref, alt]; predict both heads in one call.
batch = np.stack([one_hot_encode(ref_seq), one_hot_encode(alt_seq)], axis=0)
profile_out, counts_out = model.predict(batch, verbose=0)
profile_out = np.asarray(profile_out, dtype=np.float64)
counts_out = np.asarray(counts_out, dtype=np.float64).reshape(2, -1).sum(axis=1)

ref_lc, alt_lc = float(counts_out[0]), float(counts_out[1])
delta = alt_lc - ref_lc
jsd = jensen_shannon_distance(profile_out[0], profile_out[1])
direction = "increase" if delta > 1e-3 else "decrease" if delta < -1e-3 else "neutral"

print(f"variant       : {CHROM}:{POS}:{REF}>{ALT}  [{CELLTYPE}]")
print(f"ref log-counts: {ref_lc:+.4f}")
print(f"alt log-counts: {alt_lc:+.4f}")
print(f"Δ log-counts   : {delta:+.4f}  ({direction} accessibility)")
print(f"profile JSD    : {jsd:.4f}  (footprint-shape change)")
print("\nExpectation for rs1427407 with an erythroid model: a NEGATIVE Δ (accessibility")
print("drop) from disrupting the GATA1 motif. A positive/zero Δ => check the model cell type.")

In [ ]:
# --- OPTIONAL cross-check against RegLens's own backend ---------------------
# Once the RegLens repo is available on this Colab (upload it, or
#   !pip install -e /content/RegLens ), verify our wrapper reproduces the numbers
# above bit-for-bit — this is the real proof that the stub interface == reality.
try:
    from reglens.genome import SequenceWindow
    from reglens.tools.chrombpnet_score import (
        ChromBPNetScorer, KerasChromBPNetBackend, Variant,
    )
    window = SequenceWindow(
        chrom=CHROM, start=POS - 1 - INPUT_LEN // 2, end=POS - 1 + INPUT_LEN // 2,
        variant_offset=offset, ref_seq=ref_seq, alt_seq=alt_seq,
    )
    scorer = ChromBPNetScorer(
        KerasChromBPNetBackend(MODEL_PATH), window_length=INPUT_LEN, model_name="chrombpnet_nobias",
    )
    result = scorer.score_window(window, Variant(CHROM, POS, REF, ALT), celltype=CELLTYPE)
    print(result.summary())
    assert abs(result.delta_log_counts - delta) < 1e-6, "RegLens wrapper disagrees on Δ!"
    assert abs(result.profile_jsd - jsd) < 1e-6, "RegLens wrapper disagrees on JSD!"
    print("\n✔ RegLens KerasChromBPNetBackend reproduces the manual numbers — interface confirmed.")
except ImportError:
    print("reglens not installed on this Colab — upload the repo and `pip install -e` it to")
    print("run the cross-check. (The manual scoring above already confirms the model I/O.)")

## What a PASS looks like
- Model loads with `compile=False`; **input `(None, 2114, 4)`**, **two heads** `[profile (None, ~1000), counts (None, 1)]`.
- One variant scores end-to-end: a finite **Δ log-counts** (ideally negative for rs1427407 with an erythroid model) and a **profile JSD > 0**.
- (If the repo is present) RegLens's `KerasChromBPNetBackend` reproduces the manual Δ and JSD to `1e-6`.

## If something differs from our assumptions
- **Input length ≠ 2114** → set `window_length` in `ChromBPNetScorer` / `--window` in the CLI to the model's length.
- **Output order is `[counts, profile]`** (some exports) → pass `profile_head_index=1, counts_head_index=0` to `KerasChromBPNetBackend`.
- **Single-head (counts-only) model** → profile JSD is unavailable; `KerasChromBPNetBackend` already handles this (returns `profile_logits=None`).
- **variant-scorer parity**: it averages forward + reverse-complement predictions by default; our MVP is forward-only. Add RC-averaging later if the AUROC needs it.

Report the printed `input_shape` / `output_shape` back into `reglens/model/README.md` so the
team has the confirmed contract on record.